In [140]:
import os

In [141]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

import warnings
warnings.filterwarnings('ignore')

#**Load Dataset**
---

In [ ]:
# Load the datasets
train_df = pd.read_csv('../data/experiments/train/train_data.csv')
val_df = pd.read_csv('../data/experiments/validate/validate_data.csv')

In [143]:
train_df['Date'] = pd.to_datetime(train_df['Date'])
train_df = train_df.sort_values(['Date','Store_id'])

val_df['Date'] = pd.to_datetime(val_df['Date'])
val_df = val_df.sort_values(['Date','Store_id'])

In [144]:
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X):
        return self

    def transform(self, X):
        X = X.copy()
        X['Date'] = pd.to_datetime(X['Date'])

        X['day'] = X['Date'].dt.day
        X['month'] = X['Date'].dt.month
        X['year'] = X['Date'].dt.year
        X['weekday'] = X['Date'].dt.weekday
        X['weekofyear'] = X['Date'].dt.isocalendar().week.astype(int)
        X['is_weekend'] = X['weekday'].isin([5, 6]).astype(int)

        return X

In [145]:
class LagFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, group_col, target_cols, lags=[1, 7], rolling_windows=[7],add_aov=True):
        self.group_col = group_col
        self.target_cols = target_cols
        self.lags = lags
        self.rolling_windows = rolling_windows
        self.add_aov = add_aov

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        # Ensure proper sorting
        df = df.sort_values([self.group_col, 'Date'])

        #grouped = df.groupby(self.group_col)

        for col in self.target_cols:

            # Lags
            for lag in self.lags:
                df[f'lag_{lag}_{col}'] = (
                    df.groupby(self.group_col)[col].shift(lag)
                )

            # Rolling
            for window in self.rolling_windows:
                df[f'rolling_{col}_{window}'] = (
                    df.groupby(self.group_col)[col]
                    .transform(lambda x: x.shift(1).rolling(window).mean())
                )
        # if self.add_aov:
        #     if ('Sales' in df.columns) and ('#Order' in df.columns):

        #         sales_shift = grouped['Sales'].shift(1)
        #         order_shift = grouped['#Order'].shift(1)

        #         df['lag_1_aov'] = sales_shift / (order_shift + 1)

        #         # Optional: rolling AOV (strong feature)
        #         df['rolling_aov_7'] = (
        #             sales_shift.rolling(7).mean()
        #             /
        #             (order_shift.rolling(7).mean() + 1)
        #         )
        return df

In [146]:
# class CategoricalEncoder(BaseEstimator, TransformerMixin):
#     def __init__(self, cols):
#         self.cols = cols
#         # self.encoder = OrdinalEncoder(
#         #     handle_unknown='use_encoded_value',
#         #     unknown_value=-1
#         # )
#         self.encoder = OneHotEncoder(
#             handle_unknown='ignore',
#             sparse_output=False
#         )

#     def fit(self, X, y=None):
#         self.encoder.fit(X[self.cols])
#         return self

#     def transform(self, X):
#         X = X.copy()
#         X[self.cols] = self.encoder.transform(X[self.cols])
#         return X
    

class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.encoder = OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        )
        self.feature_names = None

    def fit(self, X, y=None):
        self.encoder.fit(X[self.cols])
        self.feature_names = self.encoder.get_feature_names_out(self.cols)
        return self

    def transform(self, X):
        X = X.copy()

        encoded = self.encoder.transform(X[self.cols])
        encoded_df = pd.DataFrame(
            encoded,
            columns=self.feature_names,
            index=X.index
        )

        #X = X.drop(columns=self.cols)
        X = pd.concat([X, encoded_df], axis=1)

        return X

In [147]:
class DerivedFeatureTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['Discount_Flag'] = (X['Discount']=='Yes').astype(int)
        X['lag_1_aov'] = X['lag_1_Sales'] / (X['lag_1_#Order'] + 1)  # Adding 1 to avoid division by zero
        return X

In [ ]:
class FeaturePipeline:
    def __init__(self):
        self.date = DateFeatureExtractor()
        self.lag = LagFeatureTransformer(
            group_col='Store_id',
            target_cols=['Sales', '#Order'],
            lags=[1, 7],
            rolling_windows=[7]
        )
        self.cat = CategoricalEncoder(
            cols=['Store_Type', 'Location_Type', 'Region_Code']
        )
        self.derived = DerivedFeatureTransformer()

    def fit_transform(self, df):
        df = self.date.fit_transform(df)
        df = self.cat.fit_transform(df)
        df = self.lag.fit_transform(df)
        df = self.derived.fit_transform(df)

        return df

    def transform(self, df):
        df = self.date.transform(df)
        df = self.cat.transform(df)
        df = self.lag.transform(df)
        df = self.derived.transform(df)

        return df

In [149]:
# class FeaturePipeline:

#     def __init__(self, lags=[1, 7], rolling_windows=[7]):
#         self.lags = lags
#         self.rolling_windows = rolling_windows
#         self.fitted = False

#     # -------------------------
#     # TIME FEATURES
#     # -------------------------
#     def create_time_features(self, df):
#         df['Date'] = pd.to_datetime(df['Date'])

#         df['day'] = df['Date'].dt.day
#         df['month'] = df['Date'].dt.month
#         df['weekday'] = df['Date'].dt.weekday
#         df['weekofyear'] = df['Date'].dt.isocalendar().week.astype(int)

#         df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)

#         return df

#     # -------------------------
#     # LAG FEATURES
#     # -------------------------
#     def create_lag_features(self, df, group_col='Store_id'):

#         for lag in self.lags:
#             df[f'lag_{lag}_#Order'] = df.groupby(group_col)['#Order'].shift(lag)
#             df[f'lag_{lag}_Sales'] = df.groupby(group_col)['Sales'].shift(lag)

#         return df

#     # -------------------------
#     # ROLLING FEATURES
#     # -------------------------
#     def create_rolling_features(self, df, group_col='Store_id'):

#         for window in self.rolling_windows:
#             df[f'rolling_#Order_{window}'] = (
#                 df.groupby(group_col)['#Order']
#                 .shift(1)
#                 .rolling(window)
#                 .mean()
#             )

#             df[f'rolling_Sales_{window}'] = (
#                 df.groupby(group_col)['Sales']
#                 .shift(1)
#                 .rolling(window)
#                 .mean()
#             )

#         return df

#     # -------------------------
#     # DERIVED FEATURES
#     # -------------------------
#     def create_derived_features(self, df):

#         df['avg_order_value'] = df['Sales'] / (df['#Order'] + 1)

#         return df

#     # -------------------------
#     # FIT (for future extensibility)
#     # -------------------------
#     def fit(self, df):
#         # Nothing to learn right now
#         # But useful if you later add encoders/scalers
#         self.fitted = True
#         return self

#     # -------------------------
#     # TRANSFORM (used in inference)
#     # -------------------------
#     def transform(self, df):

#         if not self.fitted:
#             raise ValueError("Pipeline not fitted. Call fit() first.")

#         df = df.sort_values(['Store_id', 'Date']).copy()

#         df = self.create_time_features(df)
#         df = self.create_lag_features(df)
#         df = self.create_rolling_features(df)
#         df = self.create_derived_features(df)

#         return df

#     # -------------------------
#     # FIT + TRANSFORM (training)
#     # -------------------------
#     def fit_transform(self, df):

#         df = df.sort_values(['Store_id', 'Date']).copy()

#         self.fit(df)

#         df = self.create_time_features(df)
#         df = self.create_lag_features(df)
#         df = self.create_rolling_features(df)
#         df = self.create_derived_features(df)

#         # Drop NA ONLY during training
#         #df = df.dropna()


#         return df

In [150]:
# class FeaturePipeline:

#     def __init__(self, lags=[1, 7], rolling_windows=[7]):
#         self.lags = lags
#         self.rolling_windows = rolling_windows
#         self.fitted = False
#         self.ohe = OneHotEncoder(handle_unknown='ignore')

#     def create_time_features(self, df):
#         df['Date'] = pd.to_datetime(df['Date'])

#         df['day'] = df['Date'].dt.day
#         df['month'] = df['Date'].dt.month
#         df['weekday'] = df['Date'].dt.weekday
#         df['weekofyear'] = df['Date'].dt.isocalendar().week.astype(int)
#         df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)

#         return df
    
#     def categorical_encoding(self, df):
#         df = df.copy()

#         # ---- Binary Encoding
#         df['Discount'] = df['Discount'].map({'Yes': 1, 'No': 0})

#         # ---- OneHot Encoding
#         cat_cols = ['Store_Type', 'Location_Type', 'Region_Code']

#         ohe_array = self.ohe.transform(df[cat_cols])
#         ohe_cols = self.ohe.get_feature_names_out(cat_cols)

#         ohe_df = pd.DataFrame(ohe_array, columns=ohe_cols, index=df.index)

#         df = pd.concat([df, ohe_df], axis=1)
#         df = df.drop(cat_cols, axis=1)

#         return df



#     def create_lag_features(self, df, group_col='Store_id'):
#         for lag in self.lags:
#             df[f'lag_{lag}_#Order'] = df.groupby(group_col)['#Order'].shift(lag)
#             df[f'lag_{lag}_Sales'] = df.groupby(group_col)['Sales'].shift(lag)
#         return df

#     def create_rolling_features(self, df, group_col='Store_id'):
#         for window in self.rolling_windows:
#             df[f'rolling_#Order_{window}'] = (
#                 df.groupby(group_col)['#Order']
#                 .shift(1)
#                 .rolling(window)
#                 .mean()
#             )
#             df[f'rolling_Sales_{window}'] = (
#                 df.groupby(group_col)['Sales']
#                 .shift(1)
#                 .rolling(window)
#                 .mean()
#             )
#         return df

#     def create_derived_features(self, df):
#         df['avg_order_value'] = df['Sales'] / (df['#Order'])
#         return df

#     def fit(self, df):
#         df = df.sort_values(['Store_id', 'Date']).copy()

#         df = self.create_time_features(df)

#         # ---- Fit encoder ONLY here
#         cat_cols = ['Store_Type', 'Location_Type', 'Region_Code']
#         self.ohe.fit(df[cat_cols])

#         self.fitted = True
#         return self

#     def transform(self, df):
#         if not self.fitted:
#             raise ValueError("Call fit first")

#         df = df.sort_values(['Store_id', 'Date']).copy()

#         df = self.create_time_features(df)
#         df = self.create_lag_features(df)
#         df = self.create_rolling_features(df)
#         df = self.create_derived_features(df)
#         df = self.categorical_encoding(df)

#         return df

#     def fit_transform(self, df):
#         df = df.sort_values(['Store_id', 'Date']).copy()

#         self.fit(df)

#         df = self.transform(df)

#         #df = df.dropna()

#         return df

In [151]:
train_df.shape, val_df.shape

((144540, 10), (32485, 10))

In [152]:
# feature_pipeline = Pipeline(steps=[
#     ("date", DateFeatureExtractor()),
#     ("lags", LagFeatureTransformer(
#         group_col="Store_id",
#         target_cols=["Sales", "#Order"],
#         lags=[1, 7],
#         rolling_windows=[7]
#     )),
#     ("derived", DerivedFeatureTransformer())
# ])

In [153]:
pipeline = FeaturePipeline()
train_df = pipeline.fit_transform(train_df)
val_df = pipeline.transform(val_df)

In [154]:
train_df[train_df['Store_id']==1].head(10)

,ID,Store_id,Store_Type,Location_Type,Region_Code,Date,Holiday,Discount,#Order,Sales,...,Region_Code_R3,Region_Code_R4,lag_1_Sales,lag_7_Sales,rolling_Sales_7,lag_1_#Order,lag_7_#Order,rolling_#Order_7,Discount_Flag,lag_1_aov
0,T1000001,1,S1,L3,R1,2018-01-01,1,Yes,9,7011.84,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN
365,T1000608,1,S1,L3,R1,2018-01-02,0,Yes,60,42369.00,...,0.0,0.0,7011.84,NaN,NaN,9.0,NaN,NaN,1,701.184000
730,T1001047,1,S1,L3,R1,2018-01-03,0,Yes,72,50037.00,...,0.0,0.0,42369.00,NaN,NaN,60.0,NaN,NaN,1,694.573770
1095,T1001208,1,S1,L3,R1,2018-01-04,0,Yes,64,44397.00,...,0.0,0.0,50037.00,NaN,NaN,72.0,NaN,NaN,1,685.438356
1460,T1001753,1,S1,L3,R1,2018-01-05,0,Yes,63,47604.00,...,0.0,0.0,44397.00,NaN,NaN,64.0,NaN,NaN,1,683.030769
1825,T1002163,1,S1,L3,R1,2018-01-06,0,No,36,24495.00,...,0.0,0.0,47604.00,NaN,NaN,63.0,NaN,NaN,0,743.812500
2190,T1002334,1,S1,L3,R1,2018-01-07,0,No,55,36855.00,...,0.0,0.0,24495.00,NaN,NaN,36.0,NaN,NaN,0,662.027027
2555,T1002742,1,S1,L3,R1,2018-01-08,0,No,52,34101.00,...,0.0,0.0,36855.00,7011.84,36109.834286,55.0,9.0,51.285714,0,658.125000
2920,T1003067,1,S1,L3,R1,2018-01-09,0,No,65,42429.00,...,0.0,0.0,34101.00,42369.00,39979.714286,52.0,60.0,57.428571,0,643.415094
3285,T1003300,1,S1,L3,R1,2018-01-10,0,No,53,33510.00,...,0.0,0.0,42429.00,50037.00,39988.285714,65.0,72.0,58.142857,0,642.863636


In [155]:
train_df.shape, val_df.shape

((144540, 37), (32485, 37))

In [156]:
val_df[val_df['Store_id']==1].head(10)

,ID,Store_id,Store_Type,Location_Type,Region_Code,Date,Holiday,Discount,#Order,Sales,...,Region_Code_R3,Region_Code_R4,lag_1_Sales,lag_7_Sales,rolling_Sales_7,lag_1_#Order,lag_7_#Order,rolling_#Order_7,Discount_Flag,lag_1_aov
0,T1144823,1,S1,L3,R1,2019-02-01,0,No,70,36606.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
365,T1145230,1,S1,L3,R1,2019-02-02,0,No,61,34212.0,...,0.0,0.0,36606.0,NaN,NaN,70.0,NaN,NaN,0,515.577465
730,T1145304,1,S1,L3,R1,2019-02-03,0,No,62,32640.0,...,0.0,0.0,34212.0,NaN,NaN,61.0,NaN,NaN,0,551.806452
1095,T1145889,1,S1,L3,R1,2019-02-04,0,No,33,17865.0,...,0.0,0.0,32640.0,NaN,NaN,62.0,NaN,NaN,0,518.095238
1460,T1146060,1,S1,L3,R1,2019-02-05,0,Yes,56,32028.0,...,0.0,0.0,17865.0,NaN,NaN,33.0,NaN,NaN,1,525.441176
1825,T1146653,1,S1,L3,R1,2019-02-06,0,Yes,58,32664.0,...,0.0,0.0,32028.0,NaN,NaN,56.0,NaN,NaN,1,561.894737
2190,T1147026,1,S1,L3,R1,2019-02-07,0,Yes,82,45789.0,...,0.0,0.0,32664.0,NaN,NaN,58.0,NaN,NaN,1,553.627119
2555,T1147185,1,S1,L3,R1,2019-02-08,0,Yes,58,32196.0,...,0.0,0.0,45789.0,36606.0,33114.857143,82.0,70.0,60.285714,1,551.674699
2920,T1147786,1,S1,L3,R1,2019-02-09,0,Yes,114,69162.0,...,0.0,0.0,32196.0,34212.0,32484.857143,58.0,61.0,58.571429,1,545.694915
3285,T1147852,1,S1,L3,R1,2019-02-10,1,No,106,65703.9,...,0.0,0.0,69162.0,32640.0,37477.714286,114.0,62.0,66.142857,0,601.408696


In [157]:
train_df.columns

Index(['ID', 'Store_id', 'Store_Type', 'Location_Type', 'Region_Code', 'Date',
       'Holiday', 'Discount', '#Order', 'Sales', 'day', 'month', 'year',
       'weekday', 'weekofyear', 'is_weekend', 'Store_Type_S1', 'Store_Type_S2',
       'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4', 'lag_1_Sales', 'lag_7_Sales',
       'rolling_Sales_7', 'lag_1_#Order', 'lag_7_#Order', 'rolling_#Order_7',
       'Discount_Flag', 'lag_1_aov'],
      dtype='object')

In [ ]:
train_df.to_csv('../data/experiments/train/train_transformed_data.csv', header=True, index=False)
val_df.to_csv('../data/experiments/validate/validate_transformed_data.csv', header=True, index=False)

In [159]:
feature_cols = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_#Order','lag_7_#Order','rolling_#Order_7',
    'lag_1_Sales','lag_7_Sales','rolling_Sales_7',
    'lag_1_aov',
    'Sales','#Order'
]

# 2. Calculate VIF for each variable
X = train_df[feature_cols].copy(deep=True)

# Convert categoricals if needed
X = pd.get_dummies(X, drop_first=True)
X = X.astype(float)
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna()
vif = pd.DataFrame()
vif["Feature"] = X.columns
vif["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]


In [160]:
def interpret_vif(vif_value):
    if vif_value < 5:
        return "Safe"
    elif 5 <= vif_value < 10:
        return "Moderate"
    elif 10 <= vif_value < 50:
        return "High"
    else:
        return "Severe / Potentially Perfect Collinearity"
vif['Interpretation'] = vif['VIF'].apply(interpret_vif)
vif

,Feature,VIF,Interpretation
0,Store_id,1.049296,Safe
1,Store_Type_S1,inf,Severe / Potentially Perfect Collinearity
2,Store_Type_S2,inf,Severe / Potentially Perfect Collinearity
3,Store_Type_S3,inf,Severe / Potentially Perfect Collinearity
4,Store_Type_S4,inf,Severe / Potentially Perfect Collinearity
5,Location_Type_L1,inf,Severe / Potentially Perfect Collinearity
6,Location_Type_L2,inf,Severe / Potentially Perfect Collinearity
7,Location_Type_L3,inf,Severe / Potentially Perfect Collinearity
8,Location_Type_L4,inf,Severe / Potentially Perfect Collinearity
9,Location_Type_L5,inf,Severe / Potentially Perfect Collinearity
